<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_example/02_cnn_image_classification.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

# CNN을 이용한 이미지 분류 - CIFAR-10 & Fashion-MNIST

이 노트북은 합성곱 신경망(CNN)을 사용한 이미지 분류를 학습합니다.

**학습 목표**: 
- CNN의 기본 구조와 작동 원리 이해
- 컨볼루션, 풀링, 드롭아웃 레이어의 역할
- 이미지 전처리와 데이터 증강 기법
- CNN 시각화 및 해석
- 전이 학습 기초

**Google Colab 권장**: GPU를 사용하면 훈련 속도가 크게 향상됩니다.

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
# PyTorch 임포트 (Google Colab T4 환경)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

print(f"PyTorch 버전: {torch.__version__}")

# GPU 설정 (Google Colab T4 환경)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print(f"Using device: {device}")

# 시드 설정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# 시각화 설정
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 2. 데이터셋 로드 및 탐색

In [2]:
if TENSORFLOW_AVAILABLE:
    # CIFAR-10 데이터셋 로드
    print("=== CIFAR-10 데이터셋 로드 ===")
    (X_train_cifar, y_train_cifar), (X_test_cifar, y_test_cifar) = keras.datasets.cifar10.load_data()
    
    # Fashion-MNIST 데이터셋 로드
    print("=== Fashion-MNIST 데이터셋 로드 ===")
    (X_train_fashion, y_train_fashion), (X_test_fashion, y_test_fashion) = keras.datasets.fashion_mnist.load_data()
    
    # CIFAR-10 클래스 이름
    cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                      'dog', 'frog', 'horse', 'ship', 'truck']
    
    # Fashion-MNIST 클래스 이름
    fashion_classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                      'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
    
    print(f"\nCIFAR-10:")
    print(f"  훈련 데이터: {X_train_cifar.shape}")
    print(f"  테스트 데이터: {X_test_cifar.shape}")
    print(f"  이미지 크기: {X_train_cifar.shape[1:]}")
    print(f"  클래스 수: {len(cifar10_classes)}")
    
    print(f"\nFashion-MNIST:")
    print(f"  훈련 데이터: {X_train_fashion.shape}")
    print(f"  테스트 데이터: {X_test_fashion.shape}")
    print(f"  이미지 크기: {X_train_fashion.shape[1:]}")
    print(f"  클래스 수: {len(fashion_classes)}")
else:
    print("TensorFlow를 사용할 수 없어 데이터셋을 로드할 수 없습니다.")

TensorFlow를 사용할 수 없어 데이터셋을 로드할 수 없습니다.


In [3]:
if TENSORFLOW_AVAILABLE:
    def visualize_datasets():
        """
        두 데이터셋의 샘플 이미지들을 시각화
        """
        fig, axes = plt.subplots(4, 10, figsize=(20, 8))
        
        # CIFAR-10 샘플들
        for i in range(10):
            # 각 클래스에서 첫 번째 이미지
            class_indices = np.where(y_train_cifar.flatten() == i)[0]
            sample_idx = class_indices[0]
            
            axes[0, i].imshow(X_train_cifar[sample_idx])
            axes[0, i].set_title(f'{cifar10_classes[i]}', fontsize=10)
            axes[0, i].axis('off')
            
            # 각 클래스에서 두 번째 이미지
            sample_idx = class_indices[1]
            axes[1, i].imshow(X_train_cifar[sample_idx])
            axes[1, i].axis('off')
        
        # Fashion-MNIST 샘플들
        for i in range(10):
            # 각 클래스에서 첫 번째 이미지
            class_indices = np.where(y_train_fashion.flatten() == i)[0]
            sample_idx = class_indices[0]
            
            axes[2, i].imshow(X_train_fashion[sample_idx], cmap='gray')
            axes[2, i].set_title(f'{fashion_classes[i]}', fontsize=10)
            axes[2, i].axis('off')
            
            # 각 클래스에서 두 번째 이미지
            sample_idx = class_indices[1]
            axes[3, i].imshow(X_train_fashion[sample_idx], cmap='gray')
            axes[3, i].axis('off')
        
        # 제목 설정
        axes[0, 0].text(-50, 16, 'CIFAR-10\nSamples', rotation=90, 
                       verticalalignment='center', fontsize=12, fontweight='bold')
        axes[2, 0].text(-50, 14, 'Fashion-MNIST\nSamples', rotation=90, 
                       verticalalignment='center', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    
    def analyze_dataset_properties():
        """
        데이터셋의 기본 속성 분석
        """
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # CIFAR-10 클래스 분포
        axes[0, 0].bar(range(len(cifar10_classes)), np.bincount(y_train_cifar.flatten()))
        axes[0, 0].set_title('CIFAR-10 Class Distribution')
        axes[0, 0].set_xlabel('Class')
        axes[0, 0].set_ylabel('Count')
        axes[0, 0].set_xticks(range(len(cifar10_classes)))
        axes[0, 0].set_xticklabels(cifar10_classes, rotation=45, ha='right')
        
        # Fashion-MNIST 클래스 분포
        axes[1, 0].bar(range(len(fashion_classes)), np.bincount(y_train_fashion.flatten()))
        axes[1, 0].set_title('Fashion-MNIST Class Distribution')
        axes[1, 0].set_xlabel('Class')
        axes[1, 0].set_ylabel('Count')
        axes[1, 0].set_xticks(range(len(fashion_classes)))
        axes[1, 0].set_xticklabels(fashion_classes, rotation=45, ha='right')
        
        # CIFAR-10 픽셀 값 분포
        axes[0, 1].hist(X_train_cifar.flatten(), bins=50, alpha=0.7, color='blue')
        axes[0, 1].set_title('CIFAR-10 Pixel Value Distribution')
        axes[0, 1].set_xlabel('Pixel Value')
        axes[0, 1].set_ylabel('Frequency')
        
        # Fashion-MNIST 픽셀 값 분포
        axes[1, 1].hist(X_train_fashion.flatten(), bins=50, alpha=0.7, color='green')
        axes[1, 1].set_title('Fashion-MNIST Pixel Value Distribution')
        axes[1, 1].set_xlabel('Pixel Value')
        axes[1, 1].set_ylabel('Frequency')
        
        # 평균 이미지 계산 및 시각화
        mean_cifar = np.mean(X_train_cifar, axis=0).astype(int)
        mean_fashion = np.mean(X_train_fashion, axis=0).astype(int)
        
        axes[0, 2].imshow(mean_cifar)
        axes[0, 2].set_title('CIFAR-10 Mean Image')
        axes[0, 2].axis('off')
        
        axes[1, 2].imshow(mean_fashion, cmap='gray')
        axes[1, 2].set_title('Fashion-MNIST Mean Image')
        axes[1, 2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # 통계 정보 출력
        print("\n=== 데이터셋 통계 ===")
        print(f"CIFAR-10:")
        print(f"  픽셀 값 범위: {X_train_cifar.min()} ~ {X_train_cifar.max()}")
        print(f"  평균: {X_train_cifar.mean():.2f}")
        print(f"  표준편차: {X_train_cifar.std():.2f}")
        
        print(f"\nFashion-MNIST:")
        print(f"  픽셀 값 범위: {X_train_fashion.min()} ~ {X_train_fashion.max()}")
        print(f"  평균: {X_train_fashion.mean():.2f}")
        print(f"  표준편차: {X_train_fashion.std():.2f}")
    
    # 시각화 실행
    visualize_datasets()
    analyze_dataset_properties()
else:
    print("TensorFlow를 사용할 수 없어 데이터 시각화를 건너뜁니다.")

TensorFlow를 사용할 수 없어 데이터 시각화를 건너뜁니다.


## 3. 데이터 전처리

In [4]:
if TENSORFLOW_AVAILABLE:
    def preprocess_data():
        """
        이미지 데이터 전처리
        """
        # CIFAR-10 전처리
        X_train_cifar_prep = X_train_cifar.astype('float32') / 255.0
        X_test_cifar_prep = X_test_cifar.astype('float32') / 255.0
        
        # Fashion-MNIST 전처리 (채널 차원 추가)
        X_train_fashion_prep = X_train_fashion.astype('float32') / 255.0
        X_test_fashion_prep = X_test_fashion.astype('float32') / 255.0
        X_train_fashion_prep = X_train_fashion_prep.reshape(-1, 28, 28, 1)
        X_test_fashion_prep = X_test_fashion_prep.reshape(-1, 28, 28, 1)
        
        # 원-핫 인코딩
        y_train_cifar_prep = keras.utils.to_categorical(y_train_cifar, 10)
        y_test_cifar_prep = keras.utils.to_categorical(y_test_cifar, 10)
        y_train_fashion_prep = keras.utils.to_categorical(y_train_fashion, 10)
        y_test_fashion_prep = keras.utils.to_categorical(y_test_fashion, 10)
        
        print("=== 전처리 완료 ===")
        print(f"CIFAR-10 훈련 데이터: {X_train_cifar_prep.shape}")
        print(f"CIFAR-10 레이블: {y_train_cifar_prep.shape}")
        print(f"Fashion-MNIST 훈련 데이터: {X_train_fashion_prep.shape}")
        print(f"Fashion-MNIST 레이블: {y_train_fashion_prep.shape}")
        
        return (
            X_train_cifar_prep, X_test_cifar_prep, y_train_cifar_prep, y_test_cifar_prep,
            X_train_fashion_prep, X_test_fashion_prep, y_train_fashion_prep, y_test_fashion_prep
        )
    
    # 전처리 실행
    (
        X_train_cifar_prep, X_test_cifar_prep, y_train_cifar_prep, y_test_cifar_prep,
        X_train_fashion_prep, X_test_fashion_prep, y_train_fashion_prep, y_test_fashion_prep
    ) = preprocess_data()
else:
    print("TensorFlow를 사용할 수 없어 데이터 전처리를 건너뜁니다.")

TensorFlow를 사용할 수 없어 데이터 전처리를 건너뜁니다.


## 4. 기본 CNN 모델 구축

In [5]:
if TENSORFLOW_AVAILABLE:
    def create_basic_cnn(input_shape, num_classes, model_name="Basic CNN"):
        """
        기본 CNN 모델 생성
        """
        model = models.Sequential([
            # 첫 번째 컨볼루션 블록
            layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
            layers.MaxPooling2D((2, 2)),
            
            # 두 번째 컨볼루션 블록
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            
            # 세 번째 컨볼루션 블록
            layers.Conv2D(64, (3, 3), activation='relu'),
            
            # 분류기
            layers.Flatten(),
            layers.Dense(64, activation='relu'),
            layers.Dense(num_classes, activation='softmax')
        ], name=model_name)
        
        return model
    
    def create_improved_cnn(input_shape, num_classes, model_name="Improved CNN"):
        """
        개선된 CNN 모델 (배치 정규화, 드롭아웃 포함)
        """
        model = models.Sequential([
            # 첫 번째 컨볼루션 블록
            layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
            layers.BatchNormalization(),
            layers.Conv2D(32, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # 두 번째 컨볼루션 블록
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # 세 번째 컨볼루션 블록
            layers.Conv2D(128, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.25),
            
            # 분류기
            layers.Flatten(),
            layers.Dense(512, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation='softmax')
        ], name=model_name)
        
        return model
    
    # 모델 생성
    print("=== CNN 모델 생성 ===")
    
    # CIFAR-10용 모델들
    cifar_basic_model = create_basic_cnn((32, 32, 3), 10, "CIFAR-10 Basic CNN")
    cifar_improved_model = create_improved_cnn((32, 32, 3), 10, "CIFAR-10 Improved CNN")
    
    # Fashion-MNIST용 모델들
    fashion_basic_model = create_basic_cnn((28, 28, 1), 10, "Fashion-MNIST Basic CNN")
    fashion_improved_model = create_improved_cnn((28, 28, 1), 10, "Fashion-MNIST Improved CNN")
    
    # 모델 요약 출력
    print("\n=== CIFAR-10 Basic CNN ===")
    cifar_basic_model.summary()
    
    print("\n=== CIFAR-10 Improved CNN ===")
    cifar_improved_model.summary()
else:
    print("TensorFlow를 사용할 수 없어 CNN 모델 생성을 건너뜁니다.")

TensorFlow를 사용할 수 없어 CNN 모델 생성을 건너뜁니다.


## 5. 모델 훈련 및 비교

In [6]:
if TENSORFLOW_AVAILABLE:
    def train_and_compare_models():
        """
        기본 CNN vs 개선된 CNN 성능 비교
        """
        results = {}
        
        # 모델 컴파일
        models_to_train = [
            (cifar_basic_model, X_train_cifar_prep, X_test_cifar_prep, 
             y_train_cifar_prep, y_test_cifar_prep, "CIFAR-10 Basic"),
            (cifar_improved_model, X_train_cifar_prep, X_test_cifar_prep, 
             y_train_cifar_prep, y_test_cifar_prep, "CIFAR-10 Improved"),
            (fashion_basic_model, X_train_fashion_prep, X_test_fashion_prep, 
             y_train_fashion_prep, y_test_fashion_prep, "Fashion-MNIST Basic"),
            (fashion_improved_model, X_train_fashion_prep, X_test_fashion_prep, 
             y_train_fashion_prep, y_test_fashion_prep, "Fashion-MNIST Improved")
        ]
        
        for model, X_train, X_test, y_train, y_test, model_name in models_to_train:
            print(f"\n=== {model_name} 훈련 중 ===")
            
            # 컴파일
            model.compile(
                optimizer='adam',
                loss='categorical_crossentropy',
                metrics=['accuracy']
            )
            
            # 콜백 설정
            callbacks_list = [
                callbacks.EarlyStopping(
                    monitor='val_loss',
                    patience=5,
                    restore_best_weights=True
                ),
                callbacks.ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.2,
                    patience=3
                )
            ]
            
            # 훈련 (빠른 실행을 위해 에포크 수 제한)
            history = model.fit(
                X_train, y_train,
                batch_size=128,
                epochs=10,  # 실습용으로 줄임 (실제로는 50-100 에포크 권장)
                validation_data=(X_test, y_test),
                callbacks=callbacks_list,
                verbose=1
            )
            
            # 평가
            test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
            
            results[model_name] = {
                'model': model,
                'history': history,
                'test_accuracy': test_accuracy,
                'test_loss': test_loss
            }
            
            print(f"{model_name} 테스트 정확도: {test_accuracy:.4f}")
        
        return results
    
    # 모델 훈련 실행
    print("모델 훈련을 시작합니다. 시간이 오래 걸릴 수 있습니다...")
    print("(Google Colab GPU 사용 권장)")
    
    training_results = train_and_compare_models()
else:
    print("TensorFlow를 사용할 수 없어 모델 훈련을 건너뜁니다.")

TensorFlow를 사용할 수 없어 모델 훈련을 건너뜁니다.


## 6. 훈련 결과 시각화 및 분석

In [7]:
if TENSORFLOW_AVAILABLE and 'training_results' in locals():
    def visualize_training_results(results):
        """
        훈련 결과 시각화
        """
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        
        model_names = list(results.keys())
        
        # 1. 모델별 성능 비교
        axes[0, 0].bar(model_names, [results[name]['test_accuracy'] for name in model_names])
        axes[0, 0].set_title('Test Accuracy Comparison')
        axes[0, 0].set_ylabel('Accuracy')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        # 값 표시
        for i, (name, acc) in enumerate(zip(model_names, [results[name]['test_accuracy'] for name in model_names])):
            axes[0, 0].text(i, acc + 0.01, f'{acc:.3f}', ha='center', va='bottom')
        
        # 2. 학습 곡선 - 정확도
        axes[0, 1].set_title('Training Accuracy')
        for name in model_names:
            history = results[name]['history']
            axes[0, 1].plot(history.history['accuracy'], label=f'{name} (train)', alpha=0.7)
            axes[0, 1].plot(history.history['val_accuracy'], label=f'{name} (val)', linestyle='--', alpha=0.7)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. 학습 곡선 - 손실
        axes[0, 2].set_title('Training Loss')
        for name in model_names:
            history = results[name]['history']
            axes[0, 2].plot(history.history['loss'], label=f'{name} (train)', alpha=0.7)
            axes[0, 2].plot(history.history['val_loss'], label=f'{name} (val)', linestyle='--', alpha=0.7)
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('Loss')
        axes[0, 2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[0, 2].grid(True, alpha=0.3)
        
        # 4. 모델 복잡도 비교
        model_params = [results[name]['model'].count_params() for name in model_names]
        axes[0, 3].bar(model_names, model_params)
        axes[0, 3].set_title('Model Complexity (Parameters)')
        axes[0, 3].set_ylabel('Number of Parameters')
        axes[0, 3].tick_params(axis='x', rotation=45)
        
        # 값 표시
        for i, (name, params) in enumerate(zip(model_names, model_params)):
            axes[0, 3].text(i, params + max(model_params)*0.02, f'{params:,}', 
                           ha='center', va='bottom', fontsize=8)
        
        # CIFAR-10과 Fashion-MNIST 예측 결과 분석
        datasets_info = [
            ('CIFAR-10', X_test_cifar_prep, y_test_cifar_prep, cifar10_classes),
            ('Fashion-MNIST', X_test_fashion_prep, y_test_fashion_prep, fashion_classes)
        ]
        
        for dataset_idx, (dataset_name, X_test, y_test, class_names) in enumerate(datasets_info):
            # 기본 모델과 개선된 모델 찾기
            basic_model_name = f"{dataset_name} Basic"
            improved_model_name = f"{dataset_name} Improved"
            
            if basic_model_name in results and improved_model_name in results:
                basic_model = results[basic_model_name]['model']
                improved_model = results[improved_model_name]['model']
                
                # 예측
                basic_pred = basic_model.predict(X_test, verbose=0)
                improved_pred = improved_model.predict(X_test, verbose=0)
                
                # 혼동 행렬
                basic_pred_classes = np.argmax(basic_pred, axis=1)
                improved_pred_classes = np.argmax(improved_pred, axis=1)
                true_classes = np.argmax(y_test, axis=1)
                
                # 기본 모델 혼동 행렬
                cm_basic = confusion_matrix(true_classes, basic_pred_classes)
                axes[1 + dataset_idx, 0].imshow(cm_basic, interpolation='nearest', cmap='Blues')
                axes[1 + dataset_idx, 0].set_title(f'{basic_model_name}\nConfusion Matrix')
                
                # 개선된 모델 혼동 행렬
                cm_improved = confusion_matrix(true_classes, improved_pred_classes)
                axes[1 + dataset_idx, 1].imshow(cm_improved, interpolation='nearest', cmap='Blues')
                axes[1 + dataset_idx, 1].set_title(f'{improved_model_name}\nConfusion Matrix')
                
                # 클래스별 정확도 비교
                basic_class_acc = []
                improved_class_acc = []
                
                for i in range(len(class_names)):
                    class_mask = (true_classes == i)
                    basic_class_acc.append(np.mean(basic_pred_classes[class_mask] == i))
                    improved_class_acc.append(np.mean(improved_pred_classes[class_mask] == i))
                
                x = np.arange(len(class_names))
                width = 0.35
                
                axes[1 + dataset_idx, 2].bar(x - width/2, basic_class_acc, width, 
                                            label='Basic', alpha=0.8)
                axes[1 + dataset_idx, 2].bar(x + width/2, improved_class_acc, width, 
                                            label='Improved', alpha=0.8)
                axes[1 + dataset_idx, 2].set_title(f'{dataset_name} Class-wise Accuracy')
                axes[1 + dataset_idx, 2].set_xlabel('Class')
                axes[1 + dataset_idx, 2].set_ylabel('Accuracy')
                axes[1 + dataset_idx, 2].set_xticks(x)
                axes[1 + dataset_idx, 2].set_xticklabels(class_names, rotation=45, ha='right')
                axes[1 + dataset_idx, 2].legend()
                
                # 예측 신뢰도 분석
                basic_confidence = np.max(basic_pred, axis=1)
                improved_confidence = np.max(improved_pred, axis=1)
                
                axes[1 + dataset_idx, 3].hist(basic_confidence, bins=20, alpha=0.7, 
                                             label='Basic', color='blue')
                axes[1 + dataset_idx, 3].hist(improved_confidence, bins=20, alpha=0.7, 
                                             label='Improved', color='orange')
                axes[1 + dataset_idx, 3].set_title(f'{dataset_name} Prediction Confidence')
                axes[1 + dataset_idx, 3].set_xlabel('Max Probability')
                axes[1 + dataset_idx, 3].set_ylabel('Frequency')
                axes[1 + dataset_idx, 3].legend()
        
        plt.tight_layout()
        plt.show()
    
    # 결과 시각화
    visualize_training_results(training_results)
    
    # 성능 요약
    print("\n=== 모델 성능 요약 ===")
    for name, result in training_results.items():
        print(f"{name:25}: {result['test_accuracy']:.4f} (파라미터: {result['model'].count_params():,})")
else:
    print("훈련 결과가 없어 시각화를 건너뜁니다.")

훈련 결과가 없어 시각화를 건너뜁니다.


## 7. CNN 시각화 및 해석

In [8]:
if TENSORFLOW_AVAILABLE and 'training_results' in locals():
    def visualize_cnn_features():
        """
        CNN의 특성 맵과 필터 시각화
        """
        # CIFAR-10 개선된 모델 사용
        model = training_results['CIFAR-10 Improved']['model']
        
        # 테스트 이미지 하나 선택
        test_image = X_test_cifar_prep[0:1]  # 첫 번째 이미지
        test_image_display = X_test_cifar_prep[0]
        true_label = np.argmax(y_test_cifar_prep[0])
        
        print(f"선택된 이미지의 실제 클래스: {cifar10_classes[true_label]}")
        
        # 중간층 출력을 위한 모델들 생성
        layer_outputs = [layer.output for layer in model.layers 
                        if isinstance(layer, layers.Conv2D)][:3]  # 처음 3개 Conv2D 층만
        
        if layer_outputs:
            activation_model = models.Model(inputs=model.input, outputs=layer_outputs)
            activations = activation_model.predict(test_image, verbose=0)
            
            fig, axes = plt.subplots(4, 8, figsize=(20, 10))
            
            # 원본 이미지
            axes[0, 0].imshow(test_image_display)
            axes[0, 0].set_title(f'Original\n{cifar10_classes[true_label]}')
            axes[0, 0].axis('off')
            
            # 예측 결과
            prediction = model.predict(test_image, verbose=0)
            predicted_class = np.argmax(prediction)
            confidence = np.max(prediction)
            
            axes[0, 1].bar(range(10), prediction[0])
            axes[0, 1].set_title(f'Prediction\n{cifar10_classes[predicted_class]}\n({confidence:.3f})')
            axes[0, 1].set_xticks(range(10))
            axes[0, 1].set_xticklabels([c[:3] for c in cifar10_classes], rotation=45)
            
            # 나머지 첫 번째 행 비우기
            for i in range(2, 8):
                axes[0, i].axis('off')
            
            # 각 컨볼루션 층의 특성 맵 시각화
            layer_names = ['Conv2D_1', 'Conv2D_2', 'Conv2D_3']
            
            for layer_idx, (activation, layer_name) in enumerate(zip(activations, layer_names)):
                row = layer_idx + 1
                
                # 각 층에서 처음 8개 필터의 활성화 표시
                for col in range(min(8, activation.shape[-1])):
                    axes[row, col].imshow(activation[0, :, :, col], cmap='viridis')
                    axes[row, col].set_title(f'{layer_name}\nFilter {col+1}')
                    axes[row, col].axis('off')
                
                # 남은 칸 비우기
                for col in range(min(8, activation.shape[-1]), 8):
                    axes[row, col].axis('off')
            
            plt.suptitle('CNN Feature Map Visualization', fontsize=16)
            plt.tight_layout()
            plt.show()
        
        # 첫 번째 컨볼루션 층의 필터 가중치 시각화
        conv_layer = None
        for layer in model.layers:
            if isinstance(layer, layers.Conv2D):
                conv_layer = layer
                break
        
        if conv_layer is not None:
            filters = conv_layer.get_weights()[0]
            print(f"\n첫 번째 Conv2D 층 필터 형태: {filters.shape}")
            
            # 처음 32개 필터 시각화 (8x4 grid)
            fig, axes = plt.subplots(4, 8, figsize=(16, 8))
            
            for i in range(min(32, filters.shape[-1])):
                row = i // 8
                col = i % 8
                
                # 3채널을 RGB로 정규화하여 표시
                filter_img = filters[:, :, :, i]
                filter_img = (filter_img - filter_img.min()) / (filter_img.max() - filter_img.min())
                
                axes[row, col].imshow(filter_img)
                axes[row, col].set_title(f'Filter {i+1}')
                axes[row, col].axis('off')
            
            plt.suptitle('First Convolutional Layer Filters', fontsize=16)
            plt.tight_layout()
            plt.show()
    
    def analyze_model_performance():
        """
        모델 성능 상세 분석
        """
        print("\n=== 상세 성능 분석 ===")
        
        for dataset_name in ['CIFAR-10', 'Fashion-MNIST']:
            basic_name = f"{dataset_name} Basic"
            improved_name = f"{dataset_name} Improved"
            
            if basic_name in training_results and improved_name in training_results:
                print(f"\n{dataset_name} 결과:")
                
                basic_acc = training_results[basic_name]['test_accuracy']
                improved_acc = training_results[improved_name]['test_accuracy']
                
                basic_params = training_results[basic_name]['model'].count_params()
                improved_params = training_results[improved_name]['model'].count_params()
                
                print(f"  Basic CNN    : {basic_acc:.4f} ({basic_params:,} 파라미터)")
                print(f"  Improved CNN : {improved_acc:.4f} ({improved_params:,} 파라미터)")
                print(f"  성능 향상    : {improved_acc - basic_acc:.4f} ({((improved_acc - basic_acc) / basic_acc * 100):+.1f}%)")
                print(f"  파라미터 증가: {improved_params - basic_params:,} ({((improved_params - basic_params) / basic_params * 100):+.1f}%)")
        
        # 개선 기법별 효과 분석
        print("\n=== 개선 기법들의 효과 ===")
        improvement_techniques = [
            "배치 정규화 (Batch Normalization): 훈련 안정화 및 수렴 속도 향상",
            "드롭아웃 (Dropout): 과적합 방지",
            "더 많은 필터: 더 복잡한 특성 학습",
            "더 많은 층: 더 깊은 특성 추출",
            "조기 종료 (Early Stopping): 최적 지점에서 훈련 중단",
            "학습률 스케줄링: 학습률 동적 조정"
        ]
        
        for technique in improvement_techniques:
            print(f"  • {technique}")
    
    # 시각화 및 분석 실행
    visualize_cnn_features()
    analyze_model_performance()
else:
    print("훈련된 모델이 없어 CNN 시각화를 건너뜁니다.")

훈련된 모델이 없어 CNN 시각화를 건너뜁니다.


## 8. 데이터 증강 (Data Augmentation) 실험

In [9]:
if TENSORFLOW_AVAILABLE:
    def demonstrate_data_augmentation():
        """
        데이터 증강 기법 시연 및 효과 비교
        """
        print("=== 데이터 증강 기법 시연 ===")
        
        # 데이터 증강 생성기 정의
        datagen = ImageDataGenerator(
            rotation_range=20,      # 회전
            width_shift_range=0.2,  # 수평 이동
            height_shift_range=0.2, # 수직 이동
            horizontal_flip=True,   # 수평 뒤집기
            zoom_range=0.2,         # 확대/축소
            fill_mode='nearest'     # 빈 공간 채우기
        )
        
        # 원본 이미지 선택
        sample_image = X_train_cifar_prep[0]
        sample_image = sample_image.reshape(1, 32, 32, 3)
        
        # 증강된 이미지들 생성
        fig, axes = plt.subplots(2, 5, figsize=(15, 6))
        
        # 원본 이미지
        axes[0, 0].imshow(sample_image[0])
        axes[0, 0].set_title('Original')
        axes[0, 0].axis('off')
        
        # 증강된 이미지들
        i = 1
        for batch in datagen.flow(sample_image, batch_size=1):
            row = i // 5
            col = i % 5
            
            if row < 2 and col < 5:
                axes[row, col].imshow(batch[0])
                axes[row, col].set_title(f'Augmented {i}')
                axes[row, col].axis('off')
                
                i += 1
                if i >= 10:  # 9개 증강 이미지만 생성
                    break
        
        plt.suptitle('Data Augmentation Examples', fontsize=16)
        plt.tight_layout()
        plt.show()
        
        return datagen
    
    def train_with_augmentation():
        """
        데이터 증강을 사용한 모델 훈련
        """
        print("\n=== 데이터 증강을 사용한 훈련 ===")
        
        # 데이터 증강 없는 모델
        model_no_aug = create_improved_cnn((32, 32, 3), 10, "No Augmentation")
        model_no_aug.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # 데이터 증강 있는 모델
        model_with_aug = create_improved_cnn((32, 32, 3), 10, "With Augmentation")
        model_with_aug.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # 데이터 증강 생성기
        datagen = ImageDataGenerator(
            rotation_range=15,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True,
            zoom_range=0.1
        )
        datagen.fit(X_train_cifar_prep)
        
        # 콜백 설정
        callbacks_list = [
            callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        ]
        
        # 작은 서브셋으로 빠른 비교 (실습용)
        subset_size = 5000
        X_train_subset = X_train_cifar_prep[:subset_size]
        y_train_subset = y_train_cifar_prep[:subset_size]
        
        print("\n데이터 증강 없이 훈련...")
        history_no_aug = model_no_aug.fit(
            X_train_subset, y_train_subset,
            batch_size=128,
            epochs=10,
            validation_data=(X_test_cifar_prep, y_test_cifar_prep),
            callbacks=callbacks_list,
            verbose=1
        )
        
        print("\n데이터 증강과 함께 훈련...")
        history_with_aug = model_with_aug.fit(
            datagen.flow(X_train_subset, y_train_subset, batch_size=128),
            steps_per_epoch=len(X_train_subset) // 128,
            epochs=10,
            validation_data=(X_test_cifar_prep, y_test_cifar_prep),
            callbacks=callbacks_list,
            verbose=1
        )
        
        # 결과 비교
        test_acc_no_aug = model_no_aug.evaluate(X_test_cifar_prep, y_test_cifar_prep, verbose=0)[1]
        test_acc_with_aug = model_with_aug.evaluate(X_test_cifar_prep, y_test_cifar_prep, verbose=0)[1]
        
        print(f"\n=== 결과 비교 ===")
        print(f"데이터 증강 없음: {test_acc_no_aug:.4f}")
        print(f"데이터 증강 있음: {test_acc_with_aug:.4f}")
        print(f"개선 정도: {test_acc_with_aug - test_acc_no_aug:+.4f}")
        
        # 학습 곡선 비교
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.plot(history_no_aug.history['accuracy'], label='No Aug (Train)')
        plt.plot(history_no_aug.history['val_accuracy'], label='No Aug (Val)')
        plt.plot(history_with_aug.history['accuracy'], label='With Aug (Train)')
        plt.plot(history_with_aug.history['val_accuracy'], label='With Aug (Val)')
        plt.title('Training Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 3, 2)
        plt.plot(history_no_aug.history['loss'], label='No Aug (Train)')
        plt.plot(history_no_aug.history['val_loss'], label='No Aug (Val)')
        plt.plot(history_with_aug.history['loss'], label='With Aug (Train)')
        plt.plot(history_with_aug.history['val_loss'], label='With Aug (Val)')
        plt.title('Training Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 3, 3)
        methods = ['No Augmentation', 'With Augmentation']
        accuracies = [test_acc_no_aug, test_acc_with_aug]
        colors = ['skyblue', 'lightgreen']
        
        bars = plt.bar(methods, accuracies, color=colors)
        plt.title('Final Test Accuracy')
        plt.ylabel('Accuracy')
        
        for bar, acc in zip(bars, accuracies):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{acc:.3f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        return history_no_aug, history_with_aug
    
    # 데이터 증강 실험 실행
    augmentation_datagen = demonstrate_data_augmentation()
    
    # 실제 훈련은 시간이 오래 걸리므로 선택적으로 실행
    run_augmentation_training = input("\n데이터 증강 훈련을 실행하시겠습니까? (y/n): ").lower().strip() == 'y'
    
    if run_augmentation_training:
        aug_histories = train_with_augmentation()
    else:
        print("데이터 증강 훈련을 건너뜁니다.")
else:
    print("TensorFlow를 사용할 수 없어 데이터 증강 실험을 건너뜁니다.")

TensorFlow를 사용할 수 없어 데이터 증강 실험을 건너뜁니다.


## 9. 전이 학습 기초

In [10]:
if TENSORFLOW_AVAILABLE:
    def demonstrate_transfer_learning():
        """
        VGG16을 사용한 전이 학습 기초 시연
        """
        print("=== 전이 학습 기초 시연 ===")
        print("CIFAR-10 데이터에 ImageNet으로 사전 훈련된 VGG16 사용")
        
        # CIFAR-10 이미지를 VGG16 입력 크기로 조정
        def resize_images(images, target_size=(32, 32)):
            """이미지 크기 조정 (실제로는 upsampling이 필요하지만 여기서는 간단히)"""
            # 실제 전이 학습에서는 이미지를 224x224로 리사이즈해야 하지만
            # 여기서는 개념 설명을 위해 간단한 버전을 사용
            return images
        
        # VGG16 기반 모델 생성 (작은 입력 크기를 위해 수정)
        def create_transfer_model():
            # 간단한 전이 학습 시뮬레이션
            base_model = models.Sequential([
                # VGG16 스타일의 초기 레이어들 (사전 훈련된 것처럼 가정)
                layers.Conv2D(64, (3, 3), activation='relu', input_shape=(32, 32, 3)),
                layers.Conv2D(64, (3, 3), activation='relu'),
                layers.MaxPooling2D((2, 2)),
                
                layers.Conv2D(128, (3, 3), activation='relu'),
                layers.Conv2D(128, (3, 3), activation='relu'),
                layers.MaxPooling2D((2, 2)),
                
                layers.Conv2D(256, (3, 3), activation='relu'),
                layers.MaxPooling2D((2, 2)),
            ])
            
            # 특성 추출 부분 "동결" (가중치 업데이트 안 함)
            for layer in base_model.layers[:-2]:  # 마지막 두 층 제외하고 동결
                layer.trainable = False
            
            # 새로운 분류기 추가
            model = models.Sequential([
                base_model,
                layers.Flatten(),
                layers.Dense(256, activation='relu'),
                layers.Dropout(0.5),
                layers.Dense(10, activation='softmax')  # CIFAR-10은 10개 클래스
            ])
            
            return model
        
        # 전이 학습 모델과 처음부터 훈련하는 모델 비교
        models_to_compare = {
            'From Scratch': create_improved_cnn((32, 32, 3), 10, "From Scratch"),
            'Transfer Learning': create_transfer_model()
        }
        
        # 모델 구조 비교
        print("\n=== 모델 구조 비교 ===")
        for name, model in models_to_compare.items():
            print(f"\n{name}:")
            print(f"  총 파라미터: {model.count_params():,}")
            
            trainable_params = sum([tf.size(layer.trainable_weights).numpy().sum() 
                                  for layer in model.layers if layer.trainable_weights])
            print(f"  훈련 가능한 파라미터: {trainable_params:,}")
            print(f"  동결된 파라미터: {model.count_params() - trainable_params:,}")
        
        # 전이 학습의 장점 설명
        plt.figure(figsize=(15, 10))
        
        # 개념적 비교 차트
        plt.subplot(2, 2, 1)
        methods = ['From Scratch', 'Transfer Learning']
        training_time = [100, 30]  # 상대적 훈련 시간
        data_needed = [100, 30]   # 상대적 필요 데이터량
        
        x = np.arange(len(methods))
        width = 0.35
        
        plt.bar(x - width/2, training_time, width, label='Training Time', alpha=0.8)
        plt.bar(x + width/2, data_needed, width, label='Data Needed', alpha=0.8)
        plt.xlabel('Method')
        plt.ylabel('Relative Amount')
        plt.title('Transfer Learning Advantages')
        plt.xticks(x, methods)
        plt.legend()
        
        # 전이 학습 과정 설명
        plt.subplot(2, 2, 2)
        plt.axis('off')
        
        explanation_text = """
        전이 학습 과정:
        
        1. 사전 훈련된 모델 로드
           (ImageNet 등 대용량 데이터셋)
        
        2. 특성 추출 부분 동결
           (가중치 업데이트 안 함)
        
        3. 새로운 분류기 추가
           (문제에 맞는 출력층)
        
        4. 분류기만 훈련
           (빠른 학습, 적은 데이터 필요)
        
        5. (선택적) Fine-tuning
           (전체 모델을 낮은 학습률로 미세 조정)
        """
        
        plt.text(0.1, 0.9, explanation_text, transform=plt.gca().transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.7))
        
        # 특성 맵 비교 (개념적)
        plt.subplot(2, 2, 3)
        
        # 사전 훈련된 특성들 (저수준 → 고수준)
        features = ['Edges\n& Lines', 'Shapes\n& Textures', 'Objects\n& Parts', 'Semantic\nFeatures']
        layers_depth = [1, 2, 3, 4]
        
        plt.plot(layers_depth, [1, 0.8, 0.6, 0.4], 'o-', linewidth=3, 
                markersize=10, label='Transferable Features')
        plt.plot(layers_depth, [0, 0.2, 0.4, 0.6], 's-', linewidth=3, 
                markersize=10, label='Task-specific Features')
        
        plt.xlabel('Layer Depth')
        plt.ylabel('Feature Transferability')
        plt.title('Feature Transferability by Layer')
        plt.xticks(layers_depth, features, rotation=45)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 전이 학습 전략
        plt.subplot(2, 2, 4)
        plt.axis('off')
        
        strategy_text = """
        전이 학습 전략 선택:
        
        데이터가 적고 + 유사한 문제:
        → 특성 추출기만 사용 (분류기만 훈련)
        
        데이터가 적고 + 다른 문제:
        → 낮은 층만 동결 (높은 층도 훈련)
        
        데이터가 많고 + 유사한 문제:
        → Fine-tuning (낮은 학습률로 전체 훈련)
        
        데이터가 많고 + 다른 문제:
        → 처음부터 훈련 고려
        """
        
        plt.text(0.1, 0.9, strategy_text, transform=plt.gca().transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgreen", alpha=0.7))
        
        plt.tight_layout()
        plt.show()
        
        print("\n=== 전이 학습의 장점 ===")
        advantages = [
            "• 훨씬 적은 데이터로도 좋은 성능 달성 가능",
            "• 훈련 시간 대폭 단축",
            "• 이미 학습된 저수준 특성들 활용",
            "• 과적합 위험 감소",
            "• 컴퓨팅 자원 절약"
        ]
        
        for advantage in advantages:
            print(advantage)
    
    # 전이 학습 시연
    demonstrate_transfer_learning()
else:
    print("TensorFlow를 사용할 수 없어 전이 학습 시연을 건너뜁니다.")

TensorFlow를 사용할 수 없어 전이 학습 시연을 건너뜁니다.


## 10. 실습 정리 및 핵심 개념

In [11]:
# 전체 실습 결과 요약
print("=" * 80)
print("                     CNN 이미지 분류 실습 결과 요약")
print("=" * 80)

print("\n1. 사용된 데이터셋:")
print("-" * 50)
print("• CIFAR-10: 32x32 컬러 이미지, 10개 클래스 (항공기, 자동차, 새, 고양이 등)")
print("• Fashion-MNIST: 28x28 흑백 이미지, 10개 클래스 (의류 아이템)")
print("• 각각 50,000개 훈련 이미지, 10,000개 테스트 이미지")

if TENSORFLOW_AVAILABLE and 'training_results' in locals():
    print("\n2. 모델 성능 결과:")
    print("-" * 50)
    for model_name, result in training_results.items():
        print(f"• {model_name:25}: {result['test_accuracy']:.4f}")

print("\n3. 핵심 CNN 개념들:")
print("-" * 50)
cnn_concepts = [
    "• 컨볼루션 (Convolution): 특성 추출을 위한 필터 연산",
    "• 풀링 (Pooling): 공간 차원 축소 및 위치 불변성 확보",
    "• 특성 맵 (Feature Maps): 각 필터가 감지하는 특성들",
    "• 계층적 특성 학습: 저수준 → 고수준 특성 순서로 학습",
    "• 매개변수 공유: 같은 필터를 전체 이미지에 적용",
    "• 지역 연결성: 인접한 픽셀들만 연결하여 효율성 확보"
]

for concept in cnn_concepts:
    print(concept)

print("\n4. 성능 향상 기법들:")
print("-" * 50)
improvement_techniques = [
    "• 배치 정규화: 훈련 안정화 및 수렴 속도 향상",
    "• 드롭아웃: 과적합 방지",
    "• 데이터 증강: 가상의 훈련 데이터 생성으로 일반화 성능 향상",
    "• 조기 종료: 과적합 방지를 위한 최적 지점에서 훈련 중단",
    "• 학습률 스케줄링: 훈련 진행에 따른 학습률 동적 조정",
    "• 전이 학습: 사전 훈련된 모델 활용으로 효율성 향상"
]

for technique in improvement_techniques:
    print(technique)

print("\n5. CNN vs 일반 신경망 비교:")
print("-" * 50)
comparison = [
    "• 매개변수 수: CNN이 훨씬 적음 (가중치 공유로 인해)",
    "• 공간 정보: CNN은 공간 구조 보존, 일반 NN은 평면화",
    "• 위치 불변성: CNN은 객체 위치에 덜 민감",
    "• 계산 효율성: CNN이 이미지에 특화되어 더 효율적",
    "• 해석 가능성: CNN 필터는 시각적으로 해석 가능"
]

for comp in comparison:
    print(comp)

print("\n6. 실무 적용 가이드라인:")
print("-" * 50)
guidelines = [
    "• 작은 데이터셋: 전이 학습 + 데이터 증강 활용",
    "• 큰 데이터셋: 처음부터 훈련 또는 fine-tuning",
    "• 과적합 방지: 드롭아웃 + 배치 정규화 + 조기 종료",
    "• 성능 최적화: 하이퍼파라미터 튜닝 + 앙상블",
    "• 모델 해석: 특성 맵 시각화로 학습 내용 이해",
    "• 배포 고려: 모델 압축 및 최적화 기법 적용"
]

for guideline in guidelines:
    print(guideline)

print("\n7. 다음 단계 학습 주제:")
print("-" * 50)
next_topics = [
    "• 고급 CNN 아키텍처: ResNet, DenseNet, EfficientNet",
    "• 객체 탐지: YOLO, R-CNN 계열",
    "• 시맨틱 분할: U-Net, DeepLab",
    "• 생성 모델: GAN, VAE",
    "• 어텐션 메커니즘: Vision Transformer",
    "• 모델 압축: Pruning, Quantization, Distillation"
]

for topic in next_topics:
    print(topic)

print("\n8. 환경별 실행 팁:")
print("-" * 50)

if IN_COLAB:
    print("✓ Google Colab 환경:")
    print("  • GPU 런타임 사용으로 훈련 속도 대폭 향상")
    print("  • 무료 GPU 시간 제한 주의 (12시간)")
    print("  • 대용량 데이터셋은 Google Drive 연동 활용")
else:
    print("✓ 로컬 환경:")
    if TENSORFLOW_AVAILABLE:
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            print("  • GPU 사용 가능 - 훈련 속도 향상")
        else:
            print("  • CPU 모드 - 작은 모델/데이터셋 권장")
            print("  • GPU 가속을 위해 CUDA + cuDNN 설치 고려")
    else:
        print("  • TensorFlow 설치 필요: pip install tensorflow")

print("\n" + "=" * 80)
print("                          CNN 실습 완료!")
print("=" * 80)

# 실행 환경 정보
if TENSORFLOW_AVAILABLE:
    print(f"\n실행 환경: TensorFlow {tf.__version__}")
    if tf.config.list_physical_devices('GPU'):
        print("GPU 가속 사용됨")
    else:
        print("CPU 모드로 실행됨")

                     CNN 이미지 분류 실습 결과 요약

1. 사용된 데이터셋:
--------------------------------------------------
• CIFAR-10: 32x32 컬러 이미지, 10개 클래스 (항공기, 자동차, 새, 고양이 등)
• Fashion-MNIST: 28x28 흑백 이미지, 10개 클래스 (의류 아이템)
• 각각 50,000개 훈련 이미지, 10,000개 테스트 이미지

3. 핵심 CNN 개념들:
--------------------------------------------------
• 컨볼루션 (Convolution): 특성 추출을 위한 필터 연산
• 풀링 (Pooling): 공간 차원 축소 및 위치 불변성 확보
• 특성 맵 (Feature Maps): 각 필터가 감지하는 특성들
• 계층적 특성 학습: 저수준 → 고수준 특성 순서로 학습
• 매개변수 공유: 같은 필터를 전체 이미지에 적용
• 지역 연결성: 인접한 픽셀들만 연결하여 효율성 확보

4. 성능 향상 기법들:
--------------------------------------------------
• 배치 정규화: 훈련 안정화 및 수렴 속도 향상
• 드롭아웃: 과적합 방지
• 데이터 증강: 가상의 훈련 데이터 생성으로 일반화 성능 향상
• 조기 종료: 과적합 방지를 위한 최적 지점에서 훈련 중단
• 학습률 스케줄링: 훈련 진행에 따른 학습률 동적 조정
• 전이 학습: 사전 훈련된 모델 활용으로 효율성 향상

5. CNN vs 일반 신경망 비교:
--------------------------------------------------
• 매개변수 수: CNN이 훨씬 적음 (가중치 공유로 인해)
• 공간 정보: CNN은 공간 구조 보존, 일반 NN은 평면화
• 위치 불변성: CNN은 객체 위치에 덜 민감
• 계산 효율성: CNN이 이미지에 특화되어 더 효율적
• 해석 가능성: CNN 필터는 시각적으로 해석 가능

6. 실무 적용

## 11. 추가 실험 및 연습 문제

### 연습 문제 1: 다른 데이터셋 적용
- MNIST 손글씨 숫자 데이터에 CNN 적용해보기
- CIFAR-100 (100개 클래스)에 도전해보기

### 연습 문제 2: 모델 아키텍처 실험
- 더 깊은 네트워크 (10+ 층) 만들어보기
- 잔차 연결 (Residual Connection) 추가해보기
- 다양한 필터 크기 (1x1, 5x5, 7x7) 실험

### 연습 문제 3: 고급 데이터 증강
- Cutout, Mixup 등 고급 증강 기법 구현
- 클래스별 맞춤 증강 전략 개발

### 연습 문제 4: 모델 앙상블
- 여러 CNN 모델의 앙상블 구현
- 투표 방식과 가중 평균 방식 비교

### 연습 문제 5: 실제 이미지 분류
- 웹에서 수집한 이미지로 테스트
- 실시간 웹캠 이미지 분류 시스템 구축

### 성능 최적화 팁:
1. **배치 크기 조정**: GPU 메모리에 맞게 최적화
2. **학습률 찾기**: Learning Rate Finder 사용
3. **모델 경량화**: MobileNet 스타일 아키텍처 시도
4. **하이퍼파라미터 튜닝**: Optuna, Keras Tuner 활용
5. **TensorBoard**: 훈련 과정 상세 모니터링